# Athletes Insights - Exploratory Data Analysis (EDA)
This notebook demonstrates the exploratory data analysis and cleaning process performed on the `athletes.csv` dataset for the interactive visualization dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

## 1. Loading the Dataset

In [ ]:
# Read the raw data
df = pd.read_csv('../data/athletes.csv')
print(f"Dataset Shape: {df.shape}")
df.head(3)

## 2. Inspecting Nulls and Sparsity

In [ ]:
null_counts = df.isnull().sum()
print("Missing Values by Column:")
print(null_counts[null_counts > 0])

## 3. Data Cleaning and Imputation
- Calculate athlete age relative to 2026.
- Standardize heights and weights.
- Impute missing heights and weights using gender-specific medians.

In [ ]:
# 1. Clean disciplines and events strings
for col in ['disciplines', 'events']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace("[", "", regex=False).str.replace("]", "", regex=False).str.replace("'", "", regex=False).str.replace('"', '', regex=False).str.strip()

# 2. Calculate Birth Year & Age
df['birth_date_dt'] = pd.to_datetime(df['birth_date'], errors='coerce')
fallback_years = df['birth_date'].astype(str).str.slice(0, 4).str.extract(r'(\d{4})')[0]
fallback_years = pd.to_numeric(fallback_years, errors='coerce').fillna(1996).astype(int)
birth_year = df['birth_date_dt'].dt.year.fillna(fallback_years).astype(int)
df['birth_year'] = birth_year
df['age'] = 2026 - birth_year

# 3. Group by gender and get medians for height and weight (excluding zero values)
df['height'] = df['height'].replace(0.0, np.nan)
df['weight'] = df['weight'].replace(0.0, np.nan)

medians = df.groupby('gender')[['height', 'weight']].median()
print("Gender Medians (excluding zeros):")
print(medians)

# 4. Apply gender-specific imputation
df['height'] = df.groupby('gender')['height'].transform(lambda x: x.fillna(x.median()))
df['weight'] = df.groupby('gender')['weight'].transform(lambda x: x.fillna(x.median()))

print("\nRemaining Nulls in Height & Weight:")
print(df[['height', 'weight']].isnull().sum())

## 4. Basic Demographic Analysis

In [ ]:
# Gender distribution
print(df['gender'].value_counts())

# Height and Weight statistics by gender
df.groupby('gender')[['height', 'weight', 'age']].describe()

## 5. Preliminary Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age Distribution
sns.histplot(data=df, x='age', kde=True, ax=axes[0], color='blue')
axes[0].set_title('Age Distribution of Athletes')

# Height vs Weight
sns.scatterplot(data=df, x='height', y='weight', hue='gender', alpha=0.5, ax=axes[1])
axes[1].set_title('Height vs. Weight by Gender')

plt.tight_layout()
plt.show()